In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data = {
    'name':       ['Alice','Bob','Carol','Dave','Eve','Frank','Grace','Hank','Iris','Jack',
                   'Karen','Leo','Mia','Ned','Olivia','Pete','Quinn','Rose','Sam','Tina'],
    'dept':       ['Sales','Tech','Sales','HR','Tech','Sales','Tech','HR','Sales','Tech',
                   'HR','Sales','Tech','HR','Sales','Tech','Sales','HR','Tech','Sales'],
    'age':        [25, 32, 28, 45, 36, 52, 29, 41, 33, 38, 27, 60, 31, 44, 26, 35, 48, 39, 30, 55],
    'experience': [ 2,  8,  4, 20, 12, 28,  5, 18,  9, 14,  3, 35,  7, 21,  1, 11, 24, 16,  6, 30],
    'salary':     [42, 85, 47, 61, 90, 52, 88, 63, 45, 91, 58, 54, 86, 65, 44, 89, 50, 67, 82, 200],  # 200 = outlier
    'rating':     [ 4,  5,  3,  4,  5,  3,  5,  4,  4,  5,  3,  4,  5,  3,  4,  5,  4,  3,  5,  4],
    'sales_calls':[ 40,  0, 38,  0,  0, 45,  0,  0, 42,  0,  0, 35,  0,  0, 39,  0, 44,  0,  0, 41],
    'deals_won':  [ 12,  0, 10,  0,  0, 15,  0,  0, 11,  0,  0,  9,  0,  0, 13,  0, 14,  0,  0, 12],
    'join_date':  ['2022-03-15','2016-07-01','2020-11-20','2003-05-10','2012-01-25',
                   '1996-08-14','2019-04-03','2006-02-28','2015-09-17','2010-06-12',
                   '2021-12-01','1989-03-22','2017-08-30','2003-11-05','2023-01-10',
                   '2013-07-19','2000-04-27','2008-10-03','2018-02-14','1994-06-30'],
}

df = pd.DataFrame(data)
df['join_date'] = pd.to_datetime(df['join_date'])
# salary in thousands; age/experience in years; rating 1-5; sales_calls & deals_won for Sales dept only
df.head()

# Derived Metrics

Create **new variables** from existing ones to capture more meaningful business insight.

Raw data rarely tells the full story — derived metrics translate numbers into business-relevant signals.

---

## Types of Derived Metrics

| Type | Description | Example |
|------|-------------|---------|
| **Ratio / Rate** | Divide one measure by another | Strike Rate = Runs / Balls × 100 |
| **Binary flag** | 1/0 from a threshold condition | Century = 1 if Runs ≥ 100 else 0 |
| **Aggregation** | Sum/count over a group | Total centuries per player |
| **Time extraction** | Pull year/month/day from a date | Year from MatchDate |
| **Combination** | Weighted sum of multiple columns | Total score = Maths + Reading + Science |

---

## Binary Flag from Threshold

In [ ]:
df['century'] = (df['Runs'] >= 100).astype(int)   # True→1, False→0
df['high_earner'] = (df['Salary'] > 100000).astype(int)

---

## Ratio / Rate

In [ ]:
df['strike_rate'] = (df['Runs'] / df['Balls']) * 100
df['profit_margin'] = df['Profit'] / df['Revenue']

> Always check for division by zero — filter or use `where`:
>

In [ ]:
> df['sr'] = df['Runs'].where(df['Balls'] > 0) / df['Balls'] * 100
> ```

---

## Aggregation — groupby + transform/merge

In [ ]:
# Count centuries per player
df.groupby('Player')['century'].sum().sort_values(ascending=False)

# Add group-level stat back to original df
df['player_total_centuries'] = df.groupby('Player')['century'].transform('sum')

---

## Date/Time Parsing and Extraction

In [ ]:
df['MatchDate'] = pd.to_datetime(df['MatchDate'], format='%d-%m-%Y')

# Extract components
df['Year']  = df['MatchDate'].dt.year
df['Month'] = df['MatchDate'].dt.month
df['Day']   = df['MatchDate'].dt.day

# Or all at once using apply
df[['Year','Month','Day']] = df['MatchDate'].apply(
    lambda x: pd.Series([x.year, x.month, x.day])
)

---

## Combining Columns

In [ ]:
# Sum across specific columns (ignore NaN)
subject_cols = ['Maths..', 'Reading..', 'Science..', 'Social..']
df['total_score'] = df[subject_cols].sum(axis=1)

# Weighted combination
df['score'] = 0.4 * df['Maths'] + 0.3 * df['English'] + 0.3 * df['Science']

---

## Splitting a Column into Multiple

In [ ]:
# Split "Mar-97" into Month and Year
df[['Month', 'Year']] = df['Period'].str.split('-', expand=True)

# Split on multiple delimiters or fixed positions
df['first_name'] = df['Full Name'].str.split(' ').str[0]

---

## Selecting Rows Based on a Derived Condition

In [ ]:
# Centuries only
century_df = df[df['century'] == 1].copy()

# Filter by string pattern
pattern = "Tendulkar|Lara|Ponting"
df[df['Player'].str.contains(pattern, case=False, na=False)]

---

## Workflow

1. Understand the business question
2. Identify which raw columns are inputs
3. Define the formula / logic
4. Create the column: `df['new_col'] = ...`
5. Validate: check a few rows manually, check for nulls/zeros
6. Use for segmented or bivariate analysis